# 03 – spaCy NER + Dependency Parsing

Zwei Schritte:
1. **NER:** Zahlen im Kontext erkennen (PERCENT, QUANTITY etc.)
2. **Dependency Parsing (verbessert):** Prüft ob ein *konkreter Tabellenwert*
   im selben Satz wie ein Caption-Keyword vorkommt – nicht nur irgendeine Zahl

**Setup:**
```bash
conda install -c conda-forge spacy
python -m spacy download en_core_web_sm
```

In [1]:
import json
import re
import pathlib
import pandas as pd
import spacy
from tqdm.auto import tqdm

WINDOW_SIZE = 500

try:
    nlp = spacy.load('en_core_web_md')
    print('Modell: en_core_web_md')
except OSError:
    try:
        nlp = spacy.load('en_core_web_sm')
        print('Modell: en_core_web_sm')
    except OSError:
        raise OSError('Kein spaCy-Modell.\npython -m spacy download en_core_web_sm')

notebook_path = globals().get('__vsc_ipynb_file__')
NOTEBOOK_DIR = pathlib.Path(notebook_path).resolve().parent if notebook_path else pathlib.Path.cwd()
OUTPUT_DIR = NOTEBOOK_DIR / 'output'
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = pathlib.Path.cwd() / 'output'
if not OUTPUT_DIR.exists():
    raise FileNotFoundError(f'output/ nicht gefunden')

json_files = sorted(OUTPUT_DIR.glob('*.json'))
print(f'Gefundene Dateien: {len(json_files)}')

Modell: en_core_web_md
Gefundene Dateien: 499


## Hilfsfunktionen

In [2]:
NUM_ENTS = {'CARDINAL', 'PERCENT', 'QUANTITY', 'MONEY', 'ORDINAL'}


def normalize(s):
    """Normalisiert Zahlenwerte: Komma->Punkt, Leerzeichen weg."""
    return s.strip().lower().replace(',', '.').replace(' ', '')


def extract_entities(text):
    """NER: numerische Entitäten als normalisiertes Set."""
    doc = nlp(text[:50_000])
    return {normalize(ent.text) for ent in doc.ents if ent.label_ in NUM_ENTS}


def extract_table_values(table):
    """
    Extrahiert konkrete Dezimalzahlen direkt aus den Tabellenzellen.
    Gibt normalisiertes Set zurück.
    """
    try:
        raw = table.get('content', '{}')
        content = json.loads(raw) if isinstance(raw, str) else raw
        cells = ' '.join(str(v) for row in content.get('data', []) for v in row)
    except Exception:
        return set()
    # Nur Dezimalzahlen (spezifischer, weniger Rauschen)
    return {normalize(m) for m in re.findall(r'\d+[.,]\d+', cells)}


def extract_keywords(caption):
    """Inhaltswörter aus Caption (Nomen, Adjektive, >3 Zeichen)."""
    doc = nlp((caption or '')[:300])
    return {t.lemma_.lower() for t in doc
            if t.pos_ in ('NOUN', 'PROPN', 'ADJ')
            and len(t.text) >= 4 and not t.is_stop}


def dependency_score_fixed(window_text, table_values, caption_keywords):
    """
    VERBESSERTER Dependency-Score:
    Prüft ob ein *konkreter Tabellenwert* im selben Satz
    wie ein Caption-Keyword vorkommt.

    Alter Bug: prüfte ob IRGENDEINE Zahl + Keyword im selben Satz steht
    → fast immer True in wissenschaftlichen Texten → Score zu hoch

    Neuer Ansatz: nur Treffer wenn der spezifische Tabellenwert vorkommt.
    """
    if not table_values or not caption_keywords:
        return 0.0

    doc = nlp(window_text[:10_000])
    matched_values = set()

    for sent in doc.sents:
        sent_text_norm = normalize(sent.text)
        sent_has_keyword = any(kw in sent.text.lower() for kw in caption_keywords)
        if not sent_has_keyword:
            continue
        # Taucht ein konkreter Tabellenwert in diesem Satz auf?
        for val in table_values:
            if val in sent_text_norm:
                matched_values.add(val)

    return round(len(matched_values) / len(table_values), 4) if table_values else 0.0


def get_text_window(fulltext, anchor, window=WINDOW_SIZE):
    search = anchor[:60].strip()
    if not search:
        return None
    idx = fulltext.find(search)
    if idx == -1:
        return None
    return fulltext[max(0, idx - window): idx + len(search) + window]


# Schnelltest
cap  = 'Fatty acid composition at different temperatures'
tab_vals = {'25.0', '9.4', '34.4'}  # echte Tabellenwerte
win1 = 'Palmitic acid accounted for 25.0% of total fatty acids at 10 degrees.'
win2 = 'Results were significant (p < 0.05) across all 3 conditions tested.'  # keine Tabellenwerte
kws  = extract_keywords(cap)
print('Keywords:', kws)
print('Fenster 1 (soll hoch sein):', dependency_score_fixed(win1, tab_vals, kws))
print('Fenster 2 (soll 0 sein)   :', dependency_score_fixed(win2, tab_vals, kws))

Keywords: {'composition', 'fatty', 'different', 'temperature', 'acid'}
Fenster 1 (soll hoch sein): 0.3333
Fenster 2 (soll 0 sein)   : 0.0


## Hauptanalyse

In [3]:
rows = []

for fpath in tqdm(json_files, desc='Verarbeite Dokumente'):
    with open(fpath, encoding='utf-8') as f:
        doc = json.load(f)

    year     = (doc.get('metadata') or {}).get('preprint_date', '')[:4]
    fulltext = doc.get('text', '')
    doi      = doc.get('doi', fpath.stem)
    if not year:
        continue

    for tab in doc.get('tables', []):
        if not (tab.get('caption') or tab.get('name')):
            continue
        real_refs = [r for r in (tab.get('references') or []) if len(r) > 30]
        if not real_refs:
            continue

        caption    = tab.get('caption') or tab.get('name') or ''
        table_vals = extract_table_values(tab)   # konkrete Dezimalwerte
        table_ents = extract_entities(' '.join(
            str(v) for row in (json.loads(tab['content']) if isinstance(tab.get('content'), str)
                               else (tab.get('content') or {})).get('data', []) for v in row
        )) if tab.get('content') else set()
        keywords   = extract_keywords(caption)

        if not table_vals and not table_ents:
            continue

        best_ner = 0.0
        best_dep = 0.0

        for ref in real_refs:
            window = get_text_window(fulltext, ref, WINDOW_SIZE) or ref
            window_ents = extract_entities(window)

            # NER-Overlap (wie vorher)
            if table_ents:
                ner = len(table_ents & window_ents) / len(table_ents)
                best_ner = max(best_ner, ner)

            # Verbesserter Dependency-Score
            if table_vals and keywords:
                dep = dependency_score_fixed(window, table_vals, keywords)
                best_dep = max(best_dep, dep)

        combined = round((best_ner + best_dep) / 2, 4)

        rows.append({
            'doi'        : doi,
            'year'       : year,
            'table_name' : tab.get('name', ''),
            'caption'    : caption[:80],
            'ner_overlap': round(best_ner, 4),
            'dep_score'  : round(best_dep, 4),
            'combined'   : combined,
        })

df = pd.DataFrame(rows)
print(f'Auswertbare Tabellen: {len(df)}')
if df.empty:
    raise ValueError('Keine Zeilen – output/ prüfen')
print(f'Ø NER-Overlap  : {df["ner_overlap"].mean():.4f}')
print(f'Ø Dep-Score    : {df["dep_score"].mean():.4f}  ← sollte jetzt realistischer sein')
print(f'Ø Kombiniert   : {df["combined"].mean():.4f}')
print()
print(df.groupby('year').agg(
    tabellen=('combined','count'),
    ner=('ner_overlap','mean'),
    dep=('dep_score','mean'),
    combined=('combined','mean'),
).round(4))

Verarbeite Dokumente:   0%|          | 0/499 [00:00<?, ?it/s]

Auswertbare Tabellen: 327
Ø NER-Overlap  : 0.1271
Ø Dep-Score    : 0.0693  ← sollte jetzt realistischer sein
Ø Kombiniert   : 0.0982

      tabellen     ner     dep  combined
year                                    
2022         7  0.1191  0.0000    0.0595
2023         9  0.1528  0.0135    0.0831
2024        54  0.1172  0.0901    0.1036
2025       257  0.1285  0.0688    0.0986


## Vergleich mit Regex

In [4]:
regex_csv = NOTEBOOK_DIR / 'regex_results.csv'
if regex_csv.exists():
    df_regex = pd.read_csv(regex_csv)
    merged = df.merge(df_regex[['doi','table_name','overlap']],
                      on=['doi','table_name'], how='inner')
    print(f'Gemeinsame Tabellen  : {len(merged)}')
    print(f'Ø Regex              : {merged["overlap"].mean():.4f}')
    print(f'Ø NER                : {merged["ner_overlap"].mean():.4f}')
    print(f'Ø Dep (verbessert)   : {merged["dep_score"].mean():.4f}')
    print(f'Ø Kombiniert (spaCy) : {merged["combined"].mean():.4f}')
    corr = merged['combined'].corr(merged['overlap'])
    print(f'Korrelation spaCy↔Regex: {corr:.4f}')
else:
    print('regex_results.csv nicht gefunden')

Gemeinsame Tabellen  : 278
Ø Regex              : 0.1189
Ø NER                : 0.1093
Ø Dep (verbessert)   : 0.0816
Ø Kombiniert (spaCy) : 0.0955
Korrelation spaCy↔Regex: 0.7422


In [5]:
out_csv = NOTEBOOK_DIR / 'spacy_results.csv'
df.to_csv(out_csv, index=False)
print(f'Gespeichert: {out_csv}')

Gespeichert: C:\Users\Robin\TH_Koeln\Semester_6\DIS22\spacy_results.csv
